# Setting Seeds

In [1]:
import random
from transformers import set_seed
import os

import pandas as pd
import numpy as np
import warnings
import ast
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, multilabel_confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore') 

def set_all_seeds(seed=42):
    """Set seeds for reproducible results"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set all seeds
set_all_seeds(42)

# Set device

In [2]:
if torch.backends.mps.is_available(): 
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using MPS (Apple Silicon GPU)


# Load Data and Data Preparation

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from collections import Counter
import nlpaug.augmenter.word as naw
from collections import defaultdict
import random
import re

### 1. Import clean survey data
survey_df = pd.read_excel('synthetic_data_mixed_labels.xlsx')
survey_test_df = pd.read_excel('SELT_testing_data.xlsx')

survey_test_df = survey_test_df.rename(columns={
    "pred_appropriateness": "inappropriate_appropriate",
    "pred_categories": "all_labels"
})

survey_test_df = survey_test_df[['course_name', 'question', 'student_response', 'inappropriate_appropriate', 'all_labels']].copy()

### 2. Data Preparation
# a. Student response preprocessing
def text_preprocessing(text):
    # Remove regex (any non-alphanumeric characters) and keeping only letters, numbers, and spaces using regex
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    # Lowercasing
    text = text.lower()

    return text

survey_df['student_response'] = survey_df['student_response'].apply(lambda x: text_preprocessing(x))
survey_test_df['student_response'] = survey_test_df['student_response'].apply(lambda x: text_preprocessing(x))

# b. Remove noisy text
filtered_df = survey_df[survey_df['student_response'].str.len()>10].copy()

# c. Sort and clean the label
filtered_df['all_labels'] = filtered_df['all_labels'].apply(lambda x: sorted(x) if isinstance(x, list) else x)
survey_test_df['all_labels'] = survey_test_df['all_labels'].apply(lambda x: sorted(x) if isinstance(x, list) else x)

def clean_and_sort_labels(x):
    # Convert string labels to list
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                x_list = ast.literal_eval(x)
            except:
                x_list = x.strip("[]").split(",")
        else:
            x_list = x.split(",")
    elif isinstance(x, list):
        x_list = x
    else:
        return []

    # Clean, deduplicate, and sort
    cleaned = {str(item).strip().strip("[]\"'") for item in x_list if str(item).strip().strip("[]\"'")}
    return sorted(cleaned, key=str.lower)

filtered_df['all_labels'] = filtered_df['all_labels'].apply(clean_and_sort_labels)
survey_test_df['all_labels'] = survey_test_df['all_labels'].apply(clean_and_sort_labels)


### 3. Encode the labels
# Encode multi-label
mlb = MultiLabelBinarizer(classes=["course materials & structure", "teaching & delivery", "assignment & quiz"])

y_multi = mlb.fit_transform(filtered_df["all_labels"])
y_multi_test = mlb.transform(survey_test_df["all_labels"])

# Encode binary-label
app_map = {"inappropriate": 0, "appropriate": 1}
filtered_df['inappropriate_appropriate'] = filtered_df['inappropriate_appropriate'].map(app_map)
survey_test_df['inappropriate_appropriate'] = survey_test_df['inappropriate_appropriate'].map(app_map)
y_binary_train_val = filtered_df['inappropriate_appropriate'].copy()

### 4. Split The Data
# Combine both label types into a single multi-label matrix
combined_labels = np.column_stack([y_binary_train_val.values.reshape(-1, 1), y_multi])
X = filtered_df['student_response'].copy()

# Split 1: Split out test
X_test = survey_test_df['student_response'].copy()
y_binary_test = survey_test_df['inappropriate_appropriate'].copy()

# Split 2: Split 8% of train_val as validation
X_train_val = filtered_df['student_response'].copy()
y_binary_train_val = y_binary_train_val.copy()
y_multi_train_val = y_multi.copy()

combined_labels = np.column_stack([y_binary_train_val.values.reshape(-1, 1), y_multi_train_val])
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.08, random_state=42)
train_idx, val_idx = next(msss_val.split(X_train_val, combined_labels))

X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
y_binary_train, y_binary_val = y_binary_train_val.iloc[train_idx], y_binary_train_val.iloc[val_idx]
y_multi_train, y_multi_val = y_multi_train_val[train_idx], y_multi_train_val[val_idx]

# Create DataFrames
train_df = pd.DataFrame({"student_response": X_train.values, "labels_binary": y_binary_train})
val_df = pd.DataFrame({"student_response": X_val.values, "labels_binary": y_binary_val})
test_df = pd.DataFrame({"student_response": X_test.values, "labels_binary": y_binary_test})

train_df["multi_labels"] = [list(arr) for arr in y_multi_train]
val_df["multi_labels"]   = [list(arr) for arr in y_multi_val]
test_df["multi_labels"]  = [list(arr) for arr in y_multi_test]



### 5. Check the processing result
# Train label distributions
print("Train set shape:", train_df.shape)
print(f'Train proportion label 1: {Counter(y_binary_train)}')
y_multi_train_str = ['_'.join(map(str, row)) for row in y_multi_train]
print(f'Train proportion label 2:\n{pd.Series(y_multi_train_str).value_counts()}\n')

# Validation label distributions
print("Validation set shape:", val_df.shape)
print(f'Validation proportion label 1: {Counter(y_binary_val)}')
y_multi_val_str = ['_'.join(map(str, row)) for row in y_multi_val]
print(f'Validation proportion label 2:\n{pd.Series(y_multi_val_str).value_counts()}\n')

# Test label distributions
print("Test set shape:", test_df.shape)
print(f'Test proportion label 1: {Counter(y_binary_test)}')
y_multi_test_str = ['_'.join(map(str, row)) for row in y_multi_test]
print(f'Test proportion label 2:\n{pd.Series(y_multi_test_str).value_counts()}\n')


Train set shape: (625, 3)
Train proportion label 1: Counter({1: 466, 0: 159})
Train proportion label 2:
1_0_0    159
0_1_0    145
0_0_1    139
1_1_1     78
1_1_0     49
1_0_1     32
0_1_1     23
Name: count, dtype: int64

Validation set shape: (50, 3)
Validation proportion label 1: Counter({1: 40, 0: 10})
Validation proportion label 2:
1_0_0    13
1_1_1    11
0_1_0     9
0_0_1     7
1_0_1     4
1_1_0     4
0_1_1     2
Name: count, dtype: int64

Test set shape: (100, 3)
Test proportion label 1: Counter({1: 70, 0: 30})
Test proportion label 2:
0_1_0    22
1_0_0    20
0_1_1    14
1_1_0    13
1_0_1    11
0_0_1    10
1_1_1    10
Name: count, dtype: int64



# Model Setup

In [9]:
from datasets import Sequence, Value
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
import torch.nn as nn

### 1. Load models
MODEL_NAME = "microsoft/deberta-base"

### 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

### 3. Define model Architecture
# Classifier 1: Multi-label for category prediction
multi_label_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3,
    problem_type="multi_label_classification")

multi_label_model = multi_label_model.to(device)

# Classifier 2: Binary-label for appropriateness prediction
class CascadedBinaryModel(nn.Module):
    def __init__(self, base_model_name, num_multilabel_classes=3):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.deberta.config.hidden_size
        
        # Classifier that combines text features with multi-label predictions
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_multilabel_classes, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )
        
    def forward(self, input_ids, attention_mask, multilabel_preds=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        
        # Concatenate text features with multi-label predictions
        if multilabel_preds is not None:
            combined_features = torch.cat([pooled_output, multilabel_preds], dim=1)
        else:
            # During inference without multi-label predictions
            combined_features = torch.cat([pooled_output, torch.zeros(pooled_output.size(0), 3).to(pooled_output.device)], dim=1)
            
        logits = self.classifier(combined_features)
        return type('ModelOutput', (), {'logits': logits})()

binary_model = CascadedBinaryModel(MODEL_NAME).to(device)

### 4. Freeze transformer layers
def freeze_lower_layers(model, num_layers_to_freeze=8):

    for param in model.deberta.embeddings.parameters():
        param.requires_grad = False
    
    for i in range(num_layers_to_freeze):
        for param in model.deberta.encoder.layer[i].parameters():
            param.requires_grad = False

    print(f"Frozen first {num_layers_to_freeze} layers and embeddings")
    
    # Print number of trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")

freeze_lower_layers(multi_label_model, num_layers_to_freeze=8)
freeze_lower_layers(binary_model, num_layers_to_freeze=10)

### 5. Create Hugging Face Datasets and Tokenise (updated for both label types)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize(batch):
    texts = [str(x) for x in batch["student_response"]]
    return tokenizer(texts, 
                     padding="max_length", 
                     truncation=True, 
                     max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Remove unwanted columns
keep_cols = ["input_ids", "attention_mask", "multi_labels", "labels_binary", "student_response"]

train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_cols])
val_dataset = val_dataset.remove_columns([c for c in val_dataset.column_names if c not in keep_cols])
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in keep_cols])

# Cast labels
train_dataset = train_dataset.cast_column("multi_labels", Sequence(Value("float32")))
val_dataset = val_dataset.cast_column("multi_labels", Sequence(Value("float32")))
test_dataset = test_dataset.cast_column("multi_labels", Sequence(Value("float32")))

train_dataset = train_dataset.cast_column("labels_binary", Value("int64"))
val_dataset = val_dataset.cast_column("labels_binary", Value("int64"))
test_dataset = test_dataset.cast_column("labels_binary", Value("int64"))

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "multi_labels", "labels_binary"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "multi_labels", "labels_binary"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "multi_labels", "labels_binary"])

### 6. Evaluation Metrics 
def compute_metrics_multilabel(predictions, labels, tune=False):
    """Same as your compute_metrics_multilabel but for tensors"""
    probs = torch.sigmoid(predictions).cpu().numpy()
    y_true = labels.cpu().numpy()
    
    if tune:
        # Tune thresholds per label
        best_thresholds = tune_thresholds(y_true, probs)
        y_pred = np.zeros_like(y_true)
        for i, t in enumerate(best_thresholds):
            y_pred[:, i] = (probs[:, i] >= t).astype(int)
    else:
        # Default threshold = 0.5
        y_pred = (probs >= 0.5).astype(int)
    
    # Metrics
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    subset_accuracy = (y_pred == y_true).all(axis=1).mean()
    hamming_loss = (y_pred != y_true).mean()
    
    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": hamming_loss
    }

def tune_thresholds(y_true, probs):
    best_thresholds = []
    for i in range(probs.shape[1]):
        best_f1, best_t = 0, 0.5
        for t in np.linspace(0.1, 0.9, 17):  # search 0.1 → 0.9
            preds = (probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return np.array(best_thresholds)

def compute_metrics_binary(predictions, labels):
    """Metrics for binary classification"""
    predictions = torch.sigmoid(predictions)
    predictions = (predictions > 0.5).float()
    
    predictions = predictions.cpu().numpy().flatten()
    labels = labels.cpu().numpy().flatten()
    
    
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='binary', zero_division=0)
    recall = recall_score(labels, predictions, average='binary', zero_division=0)
    f1 = f1_score(labels, predictions, average='binary', zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, predictions).ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
        "sensitivity": sensitivity
    }

### 7. Class Weighting 
# Convert multi-label column to 2D array
multi_labels_array = np.array(train_df.multi_labels.tolist())

# Calculate class weights for multi-label
def calculate_multilabel_class_weights(labels_array):
    """Calculate positive weights for each class in multi-label setting"""
    pos_weights = []
    n_samples = len(labels_array)
    
    for i in range(labels_array.shape[1]):  
        pos_count = labels_array[:, i].sum()
        neg_count = n_samples - pos_count
        
        if pos_count > 0:
            pos_weight = neg_count / pos_count
        else:
            pos_weight = 1.0
            
        pos_weights.append(pos_weight)
    
    return torch.FloatTensor(pos_weights).to(device)

# Calculate class weights for binary
def calculate_binary_class_weights(labels_array):
    """Calculate class weights for binary classification"""
    pos_count = labels_array.sum()
    neg_count = len(labels_array) - pos_count
    
    if pos_count > 0:
        pos_weight = neg_count / pos_count
    else:
        pos_weight = 1.0
        
    return torch.FloatTensor([pos_weight]).to(device)

# Calculate class weights for both models
multi_class_weights = calculate_multilabel_class_weights(multi_labels_array)
binary_class_weights = calculate_binary_class_weights(train_df.labels_binary.values)

print("\nMulti-label class weights (pos_weight):", multi_class_weights)
print("Binary class weights (pos_weight):", binary_class_weights)

# Print label distributions
labels_names = ["assignment & quiz", "course materials & structure", "teaching & delivery"]

print("\nMulti-label distribution in training data:")

for i, name in enumerate(labels_names):
    
    count = multi_labels_array[:, i].sum()
    percentage = (count / len(train_df.multi_labels)) * 100
    print(f"  {name}: {count} samples ({percentage:.1f}%)")

print(f"\nBinary label distribution:")
binary_counts = train_df.labels_binary.value_counts()
print(f"  Class 0: {binary_counts.get(0, 0)} samples")
print(f"  Class 1: {binary_counts.get(1, 0)} samples")

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Frozen first 8 layers and embeddings
Total parameters: 139,194,627
Trainable parameters: 34,449,411
Frozen parameters: 104,745,216
Frozen first 10 layers and embeddings
Total parameters: 138,898,561
Trainable parameters: 17,618,305
Frozen parameters: 121,280,256


Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Casting the dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/625 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/100 [00:00<?, ? examples/s]


Multi-label class weights (pos_weight): tensor([0.9654, 1.1186, 1.2978], device='mps:0')
Binary class weights (pos_weight): tensor([0.3412], device='mps:0')

Multi-label distribution in training data:
  assignment & quiz: 318 samples (50.9%)
  course materials & structure: 295 samples (47.2%)
  teaching & delivery: 272 samples (43.5%)

Binary label distribution:
  Class 0: 159 samples
  Class 1: 466 samples


# Training Loop

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np
from tqdm import tqdm
import os
import pandas as pd

print(f"Using device: {device}") 

def train_cascaded_model(
    multi_label_model, binary_model, train_dataset, 
    eval_dataset, multi_class_weights, binary_class_weights,
    learning_rate_multi_label=2e-5, learning_rate_binary= 3e-4,
    num_epochs_multi_label=12, num_epochs_binary = 8,
    batch_size=8, weight_decay=0.01,warmup_ratio=0.1,patience=3,
    results_dir="./results_cascaded_multi_label"):
    
    # Create results directory
    os.makedirs(results_dir, exist_ok=True)
    
    # Create data loaders
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_dataloader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
    
    # Calculate warmup steps
    warmup_steps = int(warmup_ratio * len(train_dataloader) * num_epochs_multi_label)
    
    # Loss functions
    multi_loss_fn = nn.BCEWithLogitsLoss(pos_weight=multi_class_weights)
    binary_loss_fn = nn.BCEWithLogitsLoss(pos_weight=binary_class_weights)
    
    ### Classifier 1 Training loop
    print("Classifier 1: Category Prediction")
    
    # Optimizer and scheduler for multi-label model
    multi_optimizer = torch.optim.AdamW(multi_label_model.parameters(), lr=learning_rate_multi_label, weight_decay=weight_decay)
    multi_scheduler = torch.optim.lr_scheduler.LinearLR(multi_optimizer, start_factor=0.1, total_iters=warmup_steps)
    
    # Training tracking for multi-label
    best_multi_f1_macro = 0.0
    patience_counter = 0
    all_multi_metrics = []
    
    for epoch in range(num_epochs_multi_label):
        # Training phase
        multi_label_model.train()
        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Multi-Label Epoch {epoch + 1}")
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            multi_labels = batch['multi_labels'].to(device).float()
            
            # Forward pass
            outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            loss = multi_loss_fn(logits, multi_labels)
            
            # Backward pass
            multi_optimizer.zero_grad()
            loss.backward()
            multi_optimizer.step()
            
            # Update scheduler (warmup)
            if multi_scheduler.get_last_lr()[0] < learning_rate_multi_label:
                multi_scheduler.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Evaluation phase
        multi_label_model.eval()
        eval_loss = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch in eval_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                multi_labels = batch['multi_labels'].to(device).float()
                
                outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                
                loss = multi_loss_fn(logits, multi_labels)
                eval_loss += loss.item()
                
                all_predictions.append(logits)
                all_labels.append(multi_labels)
        
        all_predictions = torch.cat(all_predictions, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        
        eval_metrics = compute_metrics_multilabel(all_predictions, all_labels)
        avg_eval_loss = eval_loss / len(eval_dataloader)
        
        # Print metrics
        print(f"Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}, Eval Loss: {avg_eval_loss:.4f} | F1 Micro: {eval_metrics['f1_micro']:.4f}, F1 Macro: {eval_metrics['f1_macro']:.4f}, Subset Acc: {eval_metrics['subset_accuracy']:.4f}")
        
        # Save metrics
        epoch_metrics = eval_metrics.copy()
        epoch_metrics.update({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'eval_loss': avg_eval_loss,
            'stage': 'multi_label'
        })
        all_multi_metrics.append(epoch_metrics)
        
        # Early stopping
        if eval_metrics['f1_macro'] > best_multi_f1_macro:
            best_multi_f1_macro = eval_metrics['f1_macro']
            patience_counter = 0
            torch.save(multi_label_model.state_dict(), f"{results_dir}/best_multilabel_model.pt")
            print(f"New best multi-label model saved! F1 Macro: {best_multi_f1_macro:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print("Early stopping triggered for multi-label model!")
                break
    
    # Load best multi-label model
    multi_label_model.load_state_dict(torch.load(f"{results_dir}/best_multilabel_model.pt"))
    
    ### Classifier 2 Training loop
    print("Classifier 2: Appropriateness Prediction")
    
    # Optimizer and scheduler for binary model
    binary_optimizer = torch.optim.AdamW(binary_model.parameters(), lr=learning_rate_binary, weight_decay=weight_decay)
    binary_scheduler = torch.optim.lr_scheduler.LinearLR(binary_optimizer, start_factor=0.1, total_iters=warmup_steps)
    
    # Training tracking for binary
    best_binary_f1 = 0.0
    patience_counter = 0
    all_binary_metrics = []
    
    for epoch in range(num_epochs_binary):
        # Training phase
        binary_model.train()
        multi_label_model.eval()  # Keep multi-label model frozen
        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Binary Epoch {epoch + 1}")
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            binary_labels = batch['labels_binary'].to(device).float().unsqueeze(1)
            
            # Get multi-label predictions (frozen)
            with torch.no_grad():
                multi_outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
                multi_preds = torch.sigmoid(multi_outputs.logits)
            
            # Forward pass through binary model
            binary_outputs = binary_model(input_ids=input_ids, attention_mask=attention_mask, multilabel_preds=multi_preds)
            binary_logits = binary_outputs.logits
            
            loss = binary_loss_fn(binary_logits, binary_labels)
            
            # Backward pass
            binary_optimizer.zero_grad()
            loss.backward()
            binary_optimizer.step()
            
            # Update scheduler
            if binary_scheduler.get_last_lr()[0] < learning_rate_binary:
                binary_scheduler.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Evaluation phase
        binary_model.eval()
        multi_label_model.eval()
        eval_loss = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch in eval_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                binary_labels = batch['labels_binary'].to(device).float().unsqueeze(1)
                
                # Get multi-label predictions
                multi_outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
                multi_preds = torch.sigmoid(multi_outputs.logits)
                
                # Get binary predictions
                binary_outputs = binary_model(input_ids=input_ids, attention_mask=attention_mask, multilabel_preds=multi_preds)
                binary_logits = binary_outputs.logits
                
                loss = binary_loss_fn(binary_logits, binary_labels)
                eval_loss += loss.item()
                
                all_predictions.append(binary_logits)
                all_labels.append(binary_labels)
        
        all_predictions = torch.cat(all_predictions, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        
        eval_metrics = compute_metrics_binary(all_predictions, all_labels)
        avg_eval_loss = eval_loss / len(eval_dataloader)
        
        # Print metrics
        print(f"Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}, Eval Loss: {avg_eval_loss:.4f} | Accuracy: {eval_metrics['accuracy']:.4f}, F1: {eval_metrics['f1']:.4f}, Precision: {eval_metrics['precision']:.4f}, Recall: {eval_metrics['recall']:.4f}")
        
        # Save metrics
        epoch_metrics = eval_metrics.copy()
        epoch_metrics.update({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'eval_loss': avg_eval_loss,
            'stage': 'binary'
        })
        all_binary_metrics.append(epoch_metrics)
        
        # Early stopping
        if eval_metrics['f1'] > best_binary_f1:
            best_binary_f1 = eval_metrics['f1']
            patience_counter = 0
            torch.save(binary_model.state_dict(), f"{results_dir}/best_binary_model.pt")
            torch.save(tokenizer, f"{results_dir}/tokenizer")
            print(f"New best binary model saved! F1: {best_binary_f1:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print("Early stopping triggered for binary model!")
                break
    
    # Create training history dataframes
    multi_history_df = pd.DataFrame(all_multi_metrics)
    multi_history_df = multi_history_df[["epoch", "train_loss", "eval_loss"] + 
                                    [c for c in multi_history_df.columns if c not in ["epoch", "train_loss", "eval_loss"]]]

    binary_history_df = pd.DataFrame(all_binary_metrics)
    binary_history_df = binary_history_df[["epoch", "train_loss", "eval_loss"] + 
                                      [c for c in binary_history_df.columns if c not in ["epoch", "train_loss", "eval_loss"]]]
    
    combined_history_df = pd.concat([multi_history_df, binary_history_df], ignore_index=True)
    
    return combined_history_df, best_multi_f1_macro, best_binary_f1

Using device: cuda


In [6]:
# Train the cascaded model
training_history_df, best_multi_f1, best_binary_f1 = train_cascaded_model(
    multi_label_model=multi_label_model,
    binary_model=binary_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    multi_class_weights=multi_class_weights,
    binary_class_weights=binary_class_weights,
    batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.1,
    patience=5,
    results_dir="./results_cascaded_multi_label"
)

print(f"\nBest Multi-Label F1 Macro: {best_multi_f1:.4f}")
print(f"Best Binary F1: {best_binary_f1:.4f}")

# Display training history
training_history_df

Classifier 1: Category Prediction


Multi-Label Epoch 1: 100%|██████████| 79/79 [00:21<00:00,  3.71it/s, loss=0.793]


Epoch 1: Train Loss: 0.7249, Eval Loss: 0.6529 | F1 Micro: 0.7283, F1 Macro: 0.7276, Subset Acc: 0.3600
New best multi-label model saved! F1 Macro: 0.7276


Multi-Label Epoch 2: 100%|██████████| 79/79 [00:20<00:00,  3.79it/s, loss=0.0788]


Epoch 2: Train Loss: 0.5410, Eval Loss: 0.4018 | F1 Micro: 0.8914, F1 Macro: 0.8896, Subset Acc: 0.6800
New best multi-label model saved! F1 Macro: 0.8896


...

Binary Epoch 5: 100%|██████████| 79/79 [00:39<00:00,  2.01it/s, loss=0.000395]


Epoch 5: Train Loss: 0.0031, Eval Loss: 0.0085 | Accuracy: 1.0000, F1: 1.0000, Precision: 1.0000, Recall: 1.0000
No improvement. Patience: 4/5


Binary Epoch 6: 100%|██████████| 79/79 [00:39<00:00,  1.98it/s, loss=4.85e-5] 


Epoch 6: Train Loss: 0.0094, Eval Loss: 0.0001 | Accuracy: 1.0000, F1: 1.0000, Precision: 1.0000, Recall: 1.0000
No improvement. Patience: 5/5
Early stopping triggered for binary model!

Best Multi-Label F1 Macro: 0.9133
Best Binary F1: 1.0000


,epoch,train_loss,eval_loss,f1_micro,f1_macro,f1_weighted,subset_accuracy,hamming_loss,stage,accuracy,precision,recall,f1
0,1,0.724889,0.652936,0.728261,0.727599,0.730134,0.36,0.333333,multi_label,NaN,NaN,NaN,NaN
1,2,0.540959,0.401767,0.891429,0.889647,0.894801,0.68,0.126667,multi_label,NaN,NaN,NaN,NaN
2,3,0.395748,0.291853,0.880952,0.879327,0.880985,0.66,0.133333,multi_label,NaN,NaN,NaN,NaN
3,4,0.296431,0.239546,0.898204,0.897148,0.898953,0.66,0.113333,multi_label,NaN,NaN,NaN,NaN
4,5,0.244354,0.267237,0.894410,0.892162,0.894606,0.70,0.113333,multi_label,NaN,NaN,NaN,NaN
5,6,0.182064,0.269899,0.894410,0.885749,0.890967,0.74,0.113333,multi_label,NaN,NaN,NaN,NaN
6,7,0.133402,0.246694,0.910180,0.908116,0.910514,0.74,0.100000,multi_label,NaN,NaN,NaN,NaN
7,8,0.102539,0.270662,0.904762,0.903438,0.905077,0.76,0.106667,multi_label,NaN,NaN,NaN,NaN
8,9,0.083552,0.259397,0.903614,0.902098,0.903994,0.70,0.106667,multi_label,NaN,NaN,NaN,NaN
9,10,0.068280,0.296714,0.919255,0.913311,0.916589,0.78,0.086667,multi_label,NaN,NaN,NaN,NaN


# Analysis

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
import seaborn as sns
import pandas as pd

def plot_cascaded_training_curves(training_history_df):
    # Separate multi-label and binary training data
    multi_df = training_history_df[training_history_df['stage'] == 'multi_label'].copy()
    binary_df = training_history_df[training_history_df['stage'] == 'binary'].copy()

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('Cascaded Model Training Progress', fontsize=16, fontweight='bold')

    ### Categorisation Loss and Metrics Plot
    if not multi_df.empty:
        epochs_multi = multi_df['epoch'].values
        train_loss_multi = multi_df['train_loss'].values
        val_loss_multi = multi_df['eval_loss'].values

        # Loss (top-left)
        axes[0, 0].plot(epochs_multi, train_loss_multi, 'b-', label='Training Loss', linewidth=2)
        axes[0, 0].plot(epochs_multi, val_loss_multi, 'r-', label='Validation Loss', linewidth=2)
        axes[0, 0].set_title('Stage 1: Multi-Label Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # Metrics (top-right)
        f1_micro = multi_df['f1_micro'].values
        f1_macro = multi_df['f1_macro'].values
        f1_weighted = multi_df['f1_weighted'].values
        subset_acc = multi_df['subset_accuracy'].values

        axes[0, 1].plot(epochs_multi, f1_micro, 'g-', label='F1 Micro', linewidth=2)
        axes[0, 1].plot(epochs_multi, f1_macro, 'b-', label='F1 Macro', linewidth=2)
        axes[0, 1].plot(epochs_multi, f1_weighted, 'purple', label='F1 Weighted', linewidth=2)
        axes[0, 1].plot(epochs_multi, subset_acc, 'orange', label='Subset Accuracy', linewidth=2)
        axes[0, 1].set_title('Stage 1: Multi-Label Metrics')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Score')
        axes[0, 1].legend()
        axes[0, 1].set_ylim(0, 1)
        axes[0, 1].grid(True, alpha=0.3)

    ### Appropriateness Loss and Metrics Plot
    if not binary_df.empty:
        epochs_binary = binary_df['epoch'].values
        train_loss_binary = binary_df['train_loss'].values
        val_loss_binary = binary_df['eval_loss'].values

        # Loss (bottom-left)
        axes[1, 0].plot(epochs_binary, train_loss_binary, 'b-', label='Training Loss', linewidth=2)
        axes[1, 0].plot(epochs_binary, val_loss_binary, 'r-', label='Validation Loss', linewidth=2)
        axes[1, 0].set_title('Stage 2: Binary Loss')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # Metrics (bottom-right)
        accuracy = binary_df['accuracy'].values
        f1 = binary_df['f1'].values
        precision = binary_df['precision'].values
        recall = binary_df['recall'].values

        axes[1, 1].plot(epochs_binary, accuracy, 'g-', label='Accuracy', linewidth=2)
        axes[1, 1].plot(epochs_binary, f1, 'b-', label='F1 Score', linewidth=2)
        axes[1, 1].plot(epochs_binary, precision, 'orange', label='Precision', linewidth=2)
        axes[1, 1].plot(epochs_binary, recall, 'red', label='Recall', linewidth=2)
        axes[1, 1].set_title('Stage 2: Binary Metrics')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Score')
        axes[1, 1].legend()
        axes[1, 1].set_ylim(0, 1)
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def get_cascaded_predictions(multi_label_model, binary_model, test_dataloader, device):
    multi_label_model.eval()
    binary_model.eval()

    all_multi_logits, all_multi_labels = [], []
    all_binary_logits, all_binary_labels = [], []
    all_binary_probs = []

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            multi_labels = batch['multi_labels'].to(device).float()
            binary_labels = batch['labels_binary'].to(device).float()

            # Multi-label forward pass
            multi_outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
            multi_logits = multi_outputs.logits
            multi_probs = torch.sigmoid(multi_logits)

            # Binary forward pass
            binary_outputs = binary_model(input_ids=input_ids, attention_mask=attention_mask, multilabel_preds=multi_probs)
            binary_logits = binary_outputs.logits
            binary_probs = torch.sigmoid(binary_logits)

            # Collect outputs
            all_multi_logits.append(multi_logits.cpu())
            all_multi_labels.append(multi_labels.cpu())
            all_binary_logits.append(binary_logits.cpu())
            all_binary_labels.append(binary_labels.cpu())
            all_binary_probs.append(binary_probs.cpu())

    # Stack results
    multi_logits = torch.cat(all_multi_logits, dim=0).numpy()
    multi_true = torch.cat(all_multi_labels, dim=0).numpy()
    binary_logits = torch.cat(all_binary_logits, dim=0).numpy().squeeze()
    binary_true = torch.cat(all_binary_labels, dim=0).numpy()
    binary_probs = torch.cat(all_binary_probs, dim=0).numpy().squeeze()

    # Threshold for multi-label
    probs = torch.sigmoid(torch.from_numpy(multi_logits)).numpy()
    multi_pred = (probs >= 0.5).astype(int)

    # Appropriateness
    binary_pred = (binary_probs >= 0.5).astype(int)

    return multi_pred, multi_true, multi_logits, binary_pred, binary_true, binary_logits, binary_probs

def plot_cascaded_confusion_matrices(multi_true, multi_pred, binary_true, binary_pred, class_names):

    app_mask = (binary_true == 1)  # Appropriate samples
    
    if app_mask.sum() > 0:
        multi_true_filtered = multi_true[app_mask]
        multi_pred_filtered = multi_pred[app_mask]
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.flatten()
        
        ### Categorisation (only appropriate samples)
        for i, class_name in enumerate(class_names[:3]):
            cm = confusion_matrix(multi_true_filtered[:, i], multi_pred_filtered[:, i])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                       xticklabels=['Not Present', 'Present'],
                       yticklabels=['Not Present', 'Present'])
            axes[i].set_title(f'Multi-Label: {class_name}\n(Appropriate Only, n={app_mask.sum()})')
            axes[i].set_xlabel('Predicted')
            axes[i].set_ylabel('Actual')
        
        ### Appropriateness
        cm_binary = confusion_matrix(binary_true, binary_pred)
        sns.heatmap(cm_binary, annot=True, fmt='d', cmap='Oranges', ax=axes[3],
                   xticklabels=['Inappropriate', 'Appropriate'],
                   yticklabels=['Inappropriate', 'Appropriate'])
        axes[3].set_title('Binary: Appropriateness')
        axes[3].set_xlabel('Predicted')
        axes[3].set_ylabel('Actual')
        
        plt.suptitle('Confusion Matrices (Multi-Label Filtered + Binary)', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print("No appropriate samples found for multi-label confusion matrices")

def plot_confidence_distributions(multi_logits, multi_true, binary_probs, binary_true, class_names):
    
    app_mask = (binary_true == 1)  # Appropriate samples
    
    if app_mask.sum() > 0:
        multi_logits_filtered = multi_logits[app_mask]
        multi_true_filtered = multi_true[app_mask]
        multi_probs_filtered = torch.sigmoid(torch.tensor(multi_logits_filtered)).numpy()

        fig, axes = plt.subplots(1, len(class_names), figsize=(12, 4))
        fig.suptitle(f'Multi-Label Confidence Distributions (Appropriate Only, n={app_mask.sum()})', 
                    fontsize=16, fontweight='bold')

        if len(class_names) == 1:
            axes = [axes] 
        
        ### Categorisation (filtered)
        for i, class_name in enumerate(class_names):
            ax = axes[i]
            pos_probs = multi_probs_filtered[multi_true_filtered[:, i] == 1, i]   
            neg_probs = multi_probs_filtered[multi_true_filtered[:, i] == 0, i]   
            
            ax.hist(neg_probs, bins=20, alpha=0.7, color='red',
                    label=f'Not {class_name}', edgecolor='black')
            ax.hist(pos_probs, bins=20, alpha=0.7, color='green',
                    label=f'{class_name}', edgecolor='black')

            ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
            ax.set_title(f'Multi-Label: {class_name}')
            ax.set_xlabel('Confidence Score')
            ax.set_ylabel('Frequency')
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print("No appropriate samples found for multi-label confidence distributions")

    # Binary confidence distribution (unchanged)
    fig, axes = plt.subplots(1, 1, figsize=(4, 4))
    fig.suptitle('Binary Confidence Distributions', fontsize=16, fontweight='bold')

    ### Appropriateness
    appropriate_probs = binary_probs[binary_true == 1]
    inappropriate_probs = binary_probs[binary_true == 0]
    axes.hist(inappropriate_probs, bins=20, alpha=0.7, color='red',
                 label='Inappropriate (True)', edgecolor='black')
    axes.hist(appropriate_probs, bins=20, alpha=0.7, color='green',
                 label='Appropriate (True)', edgecolor='black')
    axes.set_title('Binary Confidence by True Label')
    axes.set_xlabel('Confidence Score')
    axes.set_ylabel('Frequency')
    axes.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
    axes.legend()
    axes.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def print_comprehensive_report(multi_true, multi_pred, binary_true, binary_pred, class_names):
    
    app_mask = (binary_true == 1)  # Appropriate samples
    
    if app_mask.sum() > 0:
        multi_true_filtered = multi_true[app_mask]
        multi_pred_filtered = multi_pred[app_mask]
        
        print(f"\nCategorisation Classifier (Appropriate Samples Only, n={app_mask.sum()}):")    
        print(classification_report(multi_true_filtered, multi_pred_filtered, target_names=class_names, digits=4))
    else:
        print("\nCategorisation Classifier: No appropriate samples found")

    print("\nAppropriateness Classifier:") 
    print(classification_report(binary_true, binary_pred, 
                              target_names=['Inappropriate', 'Appropriate'], digits=4))
    

def visualize_cascaded_training_results(multi_label_model, binary_model, dataset, 
                                      training_history_df, device, batch_size=8):
    
    class_names = ["assignment & quiz", "course materials & structure", "teaching & delivery"]

    # Create test dataloader
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)

    (multi_pred, multi_true, multi_logits,
     binary_pred, binary_true, binary_logits, binary_probs) = get_cascaded_predictions(
        multi_label_model, binary_model, dataloader, device)
    
    # Create all visualizations
    plot_cascaded_training_curves(training_history_df)

    plot_cascaded_confusion_matrices(multi_true, multi_pred, binary_true, binary_pred, class_names)

    plot_confidence_distributions(multi_logits, multi_true, binary_probs, binary_true, class_names)

    print_comprehensive_report(multi_true, multi_pred, binary_true, binary_pred, class_names)
   

In [ ]:
visualize_cascaded_training_results(
    multi_label_model=multi_label_model,
    binary_model=binary_model, 
    dataset=val_dataset,
    training_history_df=training_history_df,
    device=device,
    batch_size=8
)

# Testing Model

In [12]:
import torch
import torch.nn as nn
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

def load_cascaded_models(results_dir="./results_cascaded_multi_label", 
                        model_name="microsoft/deberta-base",
                        device=None):

    # Check if model files exist
    multi_label_path = os.path.join(results_dir, "best_multilabel_model.pt")
    binary_path = os.path.join(results_dir, "best_binary_model.pt")
    tokenizer_path = os.path.join(results_dir, "tokenizer")
    
    if not os.path.exists(multi_label_path):
        raise FileNotFoundError(f"Multi-label model not found at: {multi_label_path}")
    if not os.path.exists(binary_path):
        raise FileNotFoundError(f"Binary model not found at: {binary_path}")
    
    print(f"Loading models from: {results_dir}")
    print(f"Using device: {device}")

    # Load tokenizer
    if os.path.exists(tokenizer_path):
        print("Loading saved tokenizer")
        try:
            # Try loading with weights_only=False for backward compatibility
            tokenizer = torch.load(tokenizer_path, weights_only=False)
        except Exception as e:
            print(f"Failed to load saved tokenizer ({e}). Loading from HuggingFace instead")
            tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    else:
        print(f"Loading tokenizer from {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    # Initialize multi-label model
    multi_label_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        problem_type="multi_label_classification"
    )
    
    # Initialize cascaded binary model
    binary_model = CascadedBinaryModel(model_name, num_multilabel_classes=3)
    
    # Load trained weights
    multi_state_dict = torch.load(multi_label_path, map_location="cpu")
    binary_state_dict = torch.load(binary_path, map_location="cpu")

    multi_label_model.load_state_dict(multi_state_dict)
    binary_model.load_state_dict(binary_state_dict)
    
    # Move models to device  
    multi_label_model.to(device)
    binary_model.to(device)

    # Set models to evaluation mode
    multi_label_model.eval()
    binary_model.eval()
    
    print("Models loaded successfully.")
    
    return multi_label_model, binary_model, tokenizer

def predict_with_cascaded_models(multi_label_model, binary_model, tokenizer, 
                                texts, multi_labels=None, binary_labels=None,
                                device=None, batch_size=8, 
                                multi_label_threshold=0.5):

    multi_label_model.eval()
    binary_model.eval()
    
    all_multi_predictions = []
    all_binary_predictions = []
    all_multi_probs = []
    all_binary_probs = []
    all_binary_logits = []
    
    # Process in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
        
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        
        with torch.no_grad():
            # Get multi-label predictions
            multi_outputs = multi_label_model(input_ids=input_ids, attention_mask=attention_mask)
            multi_logits = multi_outputs.logits
            multi_probs = torch.sigmoid(multi_logits)
            multi_preds = (multi_probs > multi_label_threshold).float()
            
            # Get binary predictions using multi-label predictions
            binary_outputs = binary_model(input_ids=input_ids, 
                                        attention_mask=attention_mask, 
                                        multilabel_preds=multi_probs)
            binary_logits = binary_outputs.logits
            binary_probs = torch.sigmoid(binary_logits)
            binary_preds = (binary_probs > 0.5).float()
            
            # Store predictions
            all_multi_predictions.extend(multi_preds.cpu().numpy())
            all_binary_predictions.extend(binary_preds.cpu().numpy())
            all_multi_probs.extend(multi_probs.cpu().numpy())
            all_binary_probs.extend(binary_probs.cpu().numpy())
            all_binary_logits.extend(binary_probs.cpu().numpy())
    
    results = {
        'multi_label_predictions': all_multi_predictions,
        'binary_predictions': all_binary_predictions,
        'multi_label_probabilities': all_multi_probs,
        'binary_probabilities': all_binary_probs,
        'binary_logits': all_binary_logits
    }
    
    # Compute metrics if labels are provided (optional)
    if binary_labels is not None:
        binary_metrics = compute_metrics_binary(
            torch.tensor(np.array(all_binary_logits), dtype=torch.float32),
            torch.tensor(np.array(binary_labels), dtype=torch.float32)
        )

        print(f"Appropriateness Classification Metrics:")
        print(f"Accuracy: {binary_metrics['accuracy']:.4f}")
        print(f"F1: {binary_metrics['f1']:.4f}")
        print(f"Precision: {binary_metrics['precision']:.4f}")
        print(f"Recall: {binary_metrics['recall']:.4f}")


        print(f"Sensitivity (True Positive Rate - Appropriate): {binary_metrics['sensitivity']:.4f}")
        print(f"Specificity (True Negative Rate - Inappropriate): {binary_metrics['specificity']:.4f}")
        
        
        results['binary_metrics'] = binary_metrics

    if multi_labels is not None and binary_labels is not None:
        
        #Filter to only appropriate samples for multi-label evaluation
        app_mask = np.array(binary_labels) == 1
        
        if app_mask.sum() > 0:
            filtered_multi_probs = np.array(all_multi_probs)
            filtered_multi_labels = np.array(multi_labels)
            
            multi_metrics = compute_metrics_multilabel(
                torch.tensor(filtered_multi_probs, dtype=torch.float32),
                torch.tensor(filtered_multi_labels, dtype=torch.float32),
                tune=True
            )

            print(f"\nCategorisation Classifier:")
            print(f"F1 Micro: {multi_metrics['f1_micro']:.4f}")
            print(f"F1 Macro: {multi_metrics['f1_macro']:.4f}")
            print(f"Subset Accuracy: {multi_metrics['subset_accuracy']:.4f}")
            results['multi_label_metrics'] = multi_metrics
        else:
            print("Categorisation Classifier: No appropriate samples found")
            results['multi_label_metrics'] = None
    
    
    if multi_labels is not None and binary_labels is not None:
        print(f"\nSummary:")
        print(f"Appropriatess F1: {binary_metrics['f1']:.4f}")
        print(f"Categorisation F1 Macro: {multi_metrics['f1_macro']:.4f}")
    
    return results


In [17]:
# Load the pre-trained model
multi_label_model, binary_model, tokenizer = load_cascaded_models(
    results_dir="./results_cascaded_multi_label/deberta",
    model_name="microsoft/deberta-base",
    device=device
)


Loading models from: ./results_cascaded_multi_label/deberta
Using device: mps
Loading saved tokenizer


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Models loaded successfully.


In [13]:
# Evaluate the prediction result using new, unseen data
results = predict_with_cascaded_models(
    multi_label_model, binary_model, tokenizer,
    texts=X_test.tolist(),
    multi_labels=y_multi_test,
    binary_labels=y_binary_test,
    device=device
)

Appropriateness Classification Metrics:
Accuracy: 0.7000
F1: 0.8235
Precision: 0.7000
Recall: 1.0000
Sensitivity (True Positive Rate - Appropriate): 1.0000
Specificity (True Negative Rate - Inappropriate): 0.0000

Categorisation Classifier:
F1 Micro: 0.6900
F1 Macro: 0.6880
Subset Accuracy: 0.1000

Summary:
Appropriatess F1: 0.8235
Categorisation F1 Macro: 0.6880


In [18]:
survey_test_df['inappropriate_appropriate'].value_counts()

inappropriate_appropriate
1    70
0    30
Name: count, dtype: int64

In [40]:
result_binary = []
for x in results['binary_predictions']:
    result_binary.append(int(x[0]))

In [ ]:
result_binary_unique = []

for x in result_binary:
    if x not in result_binary_unique:
        result_binary_unique.append(x)

In [47]:
len(result_binary)

100